In [1]:
import glob
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

In [2]:
# Get gene trait associations
RAP_DIR = 'project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results'

# ASSOC_FILE = 'loftee_mac20_associations_bh_corrected.parquet'
# ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR.parquet'
ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per.parquet'

LOCAL_DIR = '/home/dnanexus/data_dir/'

!dx download {RAP_DIR}/{ASSOC_FILE} -o {LOCAL_DIR}

gene_trait_df = (
    pl.read_parquet(f'{LOCAL_DIR}/{ASSOC_FILE}')
    .filter(pl.col('pval_fdr')<=0.05)
    .select(['region', 'phenotype', 'pval_fdr'])
)

# CORR_FILE = 'regenie_127phenotypes_mac20_lofteeHC_EUR_correlations.parquet'
# !dx download {RAP_DIR}/{CORR_FILE} -o {LOCAL_DIR}

# loftee_corrs = (
#     pl.read_parquet(f'{LOCAL_DIR}/{CORR_FILE}')
#     .with_columns(
#         loftee_corr = pl.col('correlation'),
#         loftee_corr_abs = pl.col('correlation').abs(),
#         loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
#     )
#     .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_abs', 'loftee_corr_dir']) 
# )

# gene_trait_df = (
#     gene_trait_df
#     .join(loftee_corrs, on=['region', 'phenotype'], how='inner')
#     .drop_nans()
#     .sort('loftee_corr_abs', descending=True)
#     .unique(subset=["region"], keep="first", maintain_order=True)
# )
gene_trait_df

Error: path "/home/dnanexus/data_dir/regenie_127phenotypes_lofteeHC_mac20_EUR_
miss20per.parquet" already exists but -f/--overwrite was not set


region,phenotype,pval_fdr
str,str,f64
"""ENSG00000132855""","""apolipoprotein_a_int""",0.000007
"""ENSG00000052841""","""apolipoprotein_a_int""",0.038502
"""ENSG00000110243""","""apolipoprotein_a_int""",0.003376
"""ENSG00000118137""","""apolipoprotein_a_int""",4.5099e-46
"""ENSG00000173064""","""apolipoprotein_a_int""",0.006545
…,…,…
"""ENSG00000182095""","""forced_expiratory_volume_in_1s…",0.02095
"""ENSG00000164741""","""forced_expiratory_volume_in_1s…",0.036208
"""ENSG00000205189""","""forced_expiratory_volume_in_1s…",0.045581


In [3]:
# EUR unrelated individuals

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/sample_lists/unrelated_cauc_samples_3rd_degree.csv -o /home/dnanexus/data_dir/

unrel_eur_samples = pl.read_csv('/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv')['eid'].cast(pl.Utf8).to_list()
unrel_eur_samples[:5]

Error: path "/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv"
already exists but -f/--overwrite was not set


['1000020', '1000107', '1000161', '1000172', '1000221']

In [4]:
# Download phenotypes: covariates and PRS corrected
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/phenotypes/corrected_cov_PRS_traits_EUR.parquet -o /home/dnanexus/data_dir/

phenos = (
    pl.read_parquet('/home/dnanexus/data_dir/corrected_cov_PRS_traits_EUR.parquet')
    .rename({'individual':'sample'})
    .filter(pl.col('sample').is_in(unrel_eur_samples))
)

# phenos
long_phenos = (
    phenos
    .unpivot(
        index='sample',
        on=gene_trait_df['phenotype'].unique().to_list(),
        variable_name='phenotype',
        value_name='pheno_value'
    )
    .drop_nulls()
)

print(long_phenos['phenotype'].value_counts(sort=True))
long_phenos

Error: path "/home/dnanexus/data_dir/corrected_cov_PRS_traits_EUR.parquet"
already exists but -f/--overwrite was not set
shape: (102, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u64    │
╞═════════════════════════════════╪════════╡
│ townsend_deprivation_index_at_… ┆ 378461 │
│ waist_circumference_int         ┆ 378281 │
│ hip_circumference_int           ┆ 378244 │
│ standing_height_int             ┆ 378108 │
│ weight_int                      ┆ 377841 │
│ …                               ┆ …      │
│ phosphate_int                   ┆ 330147 │
│ apolipoprotein_a_int            ┆ 328787 │
│ shbg_int                        ┆ 327605 │
│ testosterone_int                ┆ 327366 │
│ direct_bilirubin_int            ┆ 307355 │
└─────────────────────────────────┴────────┘


sample,phenotype,pheno_value
str,str,f64
"""1000020""","""mean_platelet_thrombocyte_volu…",-0.255897
"""1000107""","""mean_platelet_thrombocyte_volu…",-1.996854
"""1000161""","""mean_platelet_thrombocyte_volu…",0.150631
"""1000172""","""mean_platelet_thrombocyte_volu…",-0.376605
"""1000221""","""mean_platelet_thrombocyte_volu…",-0.667587
…,…,…
"""4974782""","""cystatin_c_int""",1.655389
"""5956310""","""cystatin_c_int""",-0.216196
"""4301443""","""cystatin_c_int""",-0.214456


In [5]:
mac = 20

RAP_ANNO_DIR = "project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"

# ANNO_FILE = "annotations_fillna_ukbgym.parquet"
ANNO_FILE = "annotations_fillna_ukbgym_with_mane.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_DIR}/{ANNO_FILE}

id_list = (
    pl.scan_parquet(f'{LOCAL_DIR}/{ANNO_FILE}')
    .filter(
        pl.col('region').is_in(gene_trait_df.select('region').unique().to_series()),
        pl.col('mac_ukb')<=mac,
    )
    .select('id')
    .unique()
    .collect()
)

id_list

Error: path
"/home/dnanexus/data_dir//annotations_fillna_ukbgym_with_mane.parquet" already
exists but -f/--overwrite was not set


/tmp/ipykernel_57338/3908164256.py:18: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


id
str
"""chr17:29387704:G:A"""
"""chr3:78664742:C:A"""
"""chr17:63062128:C:A"""
"""chr3:58156534:A:G"""
"""chr3:64660054:G:T"""
…
"""chr11:70476181:T:TCGTGCCACTGCA…"
"""chr21:21444781:C:T"""
"""chr5:79654340:A:G"""


In [6]:
# Download genotype (long gt) file
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/gt_long.parquet -o /home/dnanexus/data_dir/

long_gt = (
    pl.scan_parquet('/home/dnanexus/data_dir/gt_long.parquet')
    .select(['id', 'sample', 'gt'])
    .filter(pl.col('gt')==1)

    .join(
        id_list.lazy(),
        on='id',
        how='semi'
    )

    .collect()
)

long_gt

Error: path "/home/dnanexus/data_dir/gt_long.parquet" already exists but
-f/--overwrite was not set


id,sample,gt
str,str,i8
"""chr10:24578613:TATAAA:T""","""1745607""",1
"""chr10:24578613:TATAAA:T""","""4369337""",1
"""chr10:24578613:TATAAA:T""","""3190490""",1
"""chr10:24578613:TATAAA:T""","""2067640""",1
"""chr10:24578613:TATAAA:T""","""1906931""",1
…,…,…
"""chr9:19312532:A:C""","""4707581""",1
"""chr9:19312532:A:C""","""5902273""",1
"""chr9:19312534:A:C""","""2004709""",1


In [10]:
tmp = (
    long_phenos
    .filter(pl.col('phenotype')=='ldl_direct_int')
    .join(
        long_gt,
        on='sample',
        how='inner'
    )
    .group_by('id')
    .agg(
        n_individuals = pl.len().cast(pl.Int32),
        mean_pheno_value = pl.col('pheno_value').mean().cast(pl.Float32),
        std_pheno_value = pl.col('pheno_value').std().cast(pl.Float32),
    )
    .with_columns(
        mean_pheno_value_rank=pl.col('mean_pheno_value')
            .rank(descending=True, method="max")
            .cast(pl.Float32),
    )
    .with_columns(
        mean_pheno_value_ptile=(
            pl.col('mean_pheno_value_rank') / pl.len()
        ).cast(pl.Float32),
    )
)

tmp

id,n_individuals,mean_pheno_value,std_pheno_value,mean_pheno_value_rank,mean_pheno_value_ptile
str,i32,f32,f32,f32,f32
"""chr5:151806355:A:T""",1,-0.223121,null,1.1685805e7,0.648182
"""chr17:63159263:G:A""",15,0.017953,0.868185,8.853855e6,0.491101
"""chr5:172160829:G:C""",1,-0.57249,null,1.4592329e7,0.8094
"""chr12:21173536:T:G""",4,0.065432,1.046109,8.277671e6,0.459142
"""chr3:51086584:G:A""",2,0.04101,0.25103,8.575169e6,0.475643
…,…,…,…,…,…
"""chr2:157864700:ATCAAAT:A""",3,-0.074185,1.129137,9.979925e6,0.553562
"""chr4:147888210:T:C""",2,-0.485878,1.877692,1.401605e7,0.777435
"""chr20:3249595:G:A""",19,0.182015,0.986466,6.906027e6,0.38306


In [ ]:
output_dir = '/home/dnanexus/data_dir/appv_phenos'
!mkdir -p {output_dir}

combined_output_file = "/home/dnanexus/data_dir/loftee_mac20_quant_pheno_assocs_EURunrelated_miss20per_appv_percentiles.parquet"

combined_lazy = (
    long_phenos
    # .filter(pl.col('phenotype')=='ldl_direct_int')
    .join(
        long_gt,
        on='sample',
        how='inner'
    )
    .group_by('id')
    .agg(
        n_individuals = pl.len().cast(pl.Int32),
        mean_pheno_value = pl.col('pheno_value').mean().cast(pl.Float32),
        std_pheno_value = pl.col('pheno_value').std().cast(pl.Float32),
    )
    .with_columns(
        mean_pheno_value_rank=pl.col('mean_pheno_value')
            .rank(descending=True, method="max")
            .cast(pl.Float32),
    )
    .with_columns(
        mean_pheno_value_ptile=(
            pl.col('mean_pheno_value_rank') / pl.len()
        ).cast(pl.Float32),
    )
)

# 4. Stream to disk
print("Streaming to disk...")
combined_lazy.lazy().sink_parquet(combined_output_file, engine='streaming')
print("Done.")

In [ ]:
output_dir = '/home/dnanexus/data_dir/appv_phenos'
!mkdir -p {output_dir}

# Get the list of phenotype columns (names) for aggregation
pheno_cols = [c for c in phenos.collect_schema().names() if c != "sample"]

# sel_pheno = 'standing_height_int'
for sel_pheno in tqdm(pheno_cols):
    print(f"Processing phenotype: {sel_pheno}")
    
    (
        anno_pheno.filter(pl.col('phenotype')==sel_pheno)
        .lazy()
        .join(
            long_gt,
            on='id',
            how='inner'
        )

        .join(
            long_phenos.filter(pl.col('phenotype')==sel_pheno).lazy(),
            on=['sample', 'phenotype'],
            how='inner'
        )

        .group_by(['id', 'region', 'phenotype'])
        .agg(
            n_individuals = pl.len().cast(pl.Int32),
            mean_pheno_value = pl.col('pheno_value').mean().cast(pl.Float32),
            std_pheno_value = pl.col('pheno_value').std().cast(pl.Float32),
        )

        .with_columns(
            mean_pheno_value_rank=pl.col('mean_pheno_value')
                .rank(method="max")
                .over("region")
                .cast(pl.Float32)
        )
        .with_columns(
            mean_pheno_value_ptile=(
                pl.col('mean_pheno_value_rank') / pl.len()
            ).cast(pl.Float32)
        )

        .sink_parquet(f'{output_dir}/loftee_mac20_quant_pheno_assocs_EURunrelated_appv{sel_pheno}.parquet')
        # .collect(engine='streaming')
    )

  0%|          | 0/121 [00:00<?, ?it/s]

Processing phenotype: arm_fat_percentage_right_int


  1%|          | 1/121 [00:12<25:41, 12.84s/it]

Processing phenotype: mean_time_to_correctly_identify_matches_int


  2%|▏         | 2/121 [00:25<25:02, 12.63s/it]

...

 98%|█████████▊| 119/121 [24:39<00:24, 12.37s/it]

Processing phenotype: leg_fat_percentage_left_int


 99%|█████████▉| 120/121 [24:52<00:12, 12.35s/it]

Processing phenotype: arm_fat_mass_left_int


100%|██████████| 121/121 [25:04<00:00, 12.44s/it]


## Consolidate parquet

In [8]:
combined_output_file = "/home/dnanexus/data_dir/loftee_mac20_quant_pheno_assocs_EURunrelated_appv_percentiles.parquet"

# 1. Get list of files manually
files = glob.glob(output_dir)
print(f"Found {len(files)} files.")

# 2. Create a list of LazyFrames
lfs = [pl.scan_parquet(f) for f in files]

# 3. Concatenate with relaxation
combined_lazy = pl.concat(lfs, how="vertical_relaxed")

# 4. Stream to disk
print("Streaming to disk...")
combined_lazy.sink_parquet(combined_output_file, engine='streaming')
print("Done.")

Found 1 files.
Streaming to disk...


Done.


In [9]:
pl.scan_parquet(combined_output_file).head().collect()

id,region,phenotype,n_individuals,mean_pheno_value,std_pheno_value,mean_pheno_value_rank,mean_pheno_value_ptile
str,str,str,i32,f32,f32,f32,f32
"""chr2:47749560:C:G""","""ENSG00000116062""","""age_at_hysterectomy_int""",1,1.828922,null,12605.0,0.98886
"""chr2:47755663:C:T""","""ENSG00000116062""","""age_at_hysterectomy_int""",1,-0.102434,null,5451.0,0.42763
"""chr2:47752321:C:G""","""ENSG00000116062""","""age_at_hysterectomy_int""",1,0.446505,null,9353.0,0.733741
"""chr2:47705366:C:T""","""ENSG00000116062""","""age_at_hysterectomy_int""",2,-0.979544,1.11237,1284.0,0.10073
"""chr2:47701021:T:TAA""","""ENSG00000116062""","""age_at_hysterectomy_int""",1,0.545061,null,9847.0,0.772496
